In [46]:
from datetime import datetime, timedelta
import json
import os
import uuid

# ------------------------------
# Admin Class
# ------------------------------
class Admin:
    def __init__(self, name):
        self.name = name
        self.fine_per_day = 10

    def set_fine_rate(self, rate):
        self.fine_per_day = rate
        print(f"Admin set fine per day to Rs.{rate}")

# ------------------------------
# Book and BookCopy Classes
# ------------------------------
class Book:
    def __init__(self, title, author, isbn, subject="", publisher="", year=""):
        self.title = title
        self.author = author
        self.isbn = isbn
        self.subject = subject
        self.publisher = publisher
        self.year = year
        self.copies = []
        self.borrow_count = 0
        self.reservations = []

    def add_copy(self, copy_id, shelf="A1"):
        copy_obj = BookCopy(copy_id, self, shelf)
        self.copies.append(copy_obj)
        print(f"Added copy {copy_id} for '{self.title}' at shelf {shelf}")

    def available_copy(self):
        for c in self.copies:
            if c.available:
                return c
        return None

    def reserve(self, member):
        if member not in self.reservations:
            self.reservations.append(member)
            print(f"{member.get_name()} reserved '{self.title}'. Position in queue: {len(self.reservations)}")
        else:
            print(f"{member.get_name()} already reserved '{self.title}'.")

class BookCopy:
    def __init__(self, copy_id, parent_book, shelf="A1"):
        self.copy_id = copy_id
        self.parent_book = parent_book
        self.available = True
        self.shelf = shelf
        self.current_loan_id = None

# ------------------------------
# Transaction Class
# ------------------------------
class Transaction:
    def __init__(self, action, member_id, details="", amount=0.0):
        self.txn_id = str(uuid.uuid4())
        self.action = action
        self.member_id = member_id
        self.amount = amount
        self.details = details
        self.date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.save_transaction()

    def save_transaction(self):
        try:
            with open("transactions.txt", "a") as f:
                f.write(f"{self.date},{self.member_id},{self.action},{self.details},{self.amount}\n")
        except Exception as e:
            print("Error saving transaction:", e)

# ------------------------------
# Member Classes
# ------------------------------
class Member:
    def __init__(self, name, contact="", pin=1234):
        self.__member_id = str(uuid.uuid4())        
        self.__name = name
        self.__contact = contact
        self.__pin = pin
        self.pin_attempts = 0
        self.locked = False
        self.approved = False
        self.loans = []
        self.fines = 0.0
        self.borrow_limit = 3
        self.loan_days = 14

    def get_name(self):
        return self.__name

    def get_contact(self):
        return self.__contact

    def get_member_id(self):
        return self.__member_id

    def get_fines(self):
        return self.fines

    def is_locked(self):
        return self.locked
    
    def is_approved(self):
        return self.approved

    def verify_pin(self, pin):
        if self.locked:
            print("PIN attempts exceeded — account locked.")
            return False
        if pin == self.__pin:
            self.pin_attempts = 0
            return True
        else:
            self.pin_attempts += 1
            if self.pin_attempts >= 3:
                self.locked = True
                print("PIN attempts exceeded — account locked for review.")
            else:
                print(f"Incorrect PIN. {3 - self.pin_attempts} attempts left.")
            return False

class StudentMember(Member):
    def __init__(self, name, contact="", pin=1234):
        super().__init__(name, contact, pin)
        self.borrow_limit = 5
        self.loan_days = 14
        self.renewals_allowed = 1

class FacultyMember(Member):
    def __init__(self, name, contact="", pin=1234):
        super().__init__(name, contact, pin)
        self.borrow_limit = 10
        self.loan_days = 28
        self.renewals_allowed = 2

class GuestMember(Member):
    def __init__(self, name, contact="", pin=1234):
        super().__init__(name, contact, pin)
        self.borrow_limit = 2
        self.loan_days = 7
        self.renewals_allowed = 0

class Librarian(Member):
    def __init__(self, name, pin=9999):
        super().__init__(name, "N/A", pin)
        self.approved = True

    def approve_member(self, member):
        member.approved = True
        print(f"Librarian approved member {member.get_member_id()} : {member.get_name()}")

    def reset_lock(self, member):
        member.locked = False
        member.pin_attempts = 0
        print(f"Librarian reset lock for {member.get_member_id()}")

# ------------------------------
# Loan & Fine Classes
# ------------------------------
class Loan:
    def __init__(self, member, copy_obj):
        self.loan_id = str(uuid.uuid4())
        self.member = member
        self.copy_obj = copy_obj
        self.date_borrowed = datetime.now()
        self.due_date = self.date_borrowed + timedelta(days=member.loan_days)
        self.return_date = None
        self.renewals = 0

    def renew(self):
        book = self.copy_obj.parent_book
        if book.reservations and book.reservations[0] != self.member:
            print("Cannot renew — another member has reserved this book.")
            return False
        if self.renewals >= getattr(self.member, "renewals_allowed", 0):
            print("Renewal limit reached for this member.")
            return False
        self.due_date += timedelta(days=self.member.loan_days)
        self.renewals += 1
        print(f"Renewed loan {self.loan_id}. New due date: {self.due_date.strftime('%Y-%m-%d')}")
        return True

    def days_overdue(self):
        diff = (datetime.now() - self.due_date).days
        return diff if diff > 0 else 0

class Fine:
    def __init__(self, loan_id, member_id, amount):
        self.fine_id = str(uuid.uuid4())
        self.loan_id = loan_id
        self.member_id = member_id
        self.amount = amount
        self.date = datetime.now().strftime("%Y-%m-%d")
        self.paid = False
        self.save_fine()

    def save_fine(self):
        try:
            if os.path.exists("fines.json"):
                with open("fines.json", "r") as f:
                    fines_list = json.load(f)
            else:
                fines_list = []
            fines_list.append({
                "fine_id": self.fine_id,
                "loan_id": self.loan_id,
                "member_id": self.member_id,
                "amount": self.amount,
                "paid": self.paid,
                "date": self.date
            })
            with open("fines.json", "w") as f:
                json.dump(fines_list, f)
        except Exception as e:
            print("Error saving fine:", e)

# ------------------------------
# Reservation Manager
# ------------------------------
class ReservationManager:
    def __init__(self):
        self.queues = {}

    def create_reservation(self, book, member):
        isbn = book.isbn
        if isbn not in self.queues:
            self.queues[isbn] = []
        if member not in self.queues[isbn]:
            self.queues[isbn].append(member)
            book.reserve(member)
            print(f"Reservation created for {member.get_name()} on '{book.title}'")
            Transaction("reserve", member.get_member_id(), f"Book {book.title}")
        else:
            print(f"{member.get_name()} already in reservation queue for '{book.title}'")

    def cancel_reservation(self, book, member):
        isbn = book.isbn
        if isbn in self.queues and member in self.queues[isbn]:
            self.queues[isbn].remove(member)
            if member in book.reservations:
                book.reservations.remove(member)
            print(f"Reservation cancelled for {member.get_name()} on '{book.title}'")

# ------------------------------
# Loan Manager
# ------------------------------
class LoanManager:
    def __init__(self, admin):
        self.active_loans = []
        self.loan_history = []
        self.admin = admin

    def create_loan(self, member, copy_obj):
        if len(member.loans) >= member.borrow_limit:
            print(f"Member {member.get_name()} has reached borrow limit.")
            return None
        loan = Loan(member, copy_obj)
        copy_obj.available = False
        copy_obj.current_loan_id = loan.loan_id
        member.loans.append(loan)
        self.active_loans.append(loan)
        self.loan_history.append(loan)
        print(f"Loan created {loan.loan_id} for {member.get_name()} copy {copy_obj.copy_id}")
        Transaction("borrow", member.get_member_id(), f"Book {copy_obj.parent_book.title}")
        copy_obj.parent_book.borrow_count += 1
        return loan

    def return_loan(self, loan):
        loan.return_date = datetime.now()
        days = loan.days_overdue()
        fine_amount = 0
        if days > 0:
            fine_amount = days * self.admin.fine_per_day
            loan.member.fines += fine_amount
            Fine(loan.loan_id, loan.member.get_member_id(), fine_amount)
            print(f"Loan {loan.loan_id} returned late by {days} days. Fine Rs.{fine_amount}")
        else:
            print(f"Loan {loan.loan_id} returned on time.")
        loan.copy_obj.available = True
        loan.copy_obj.current_loan_id = None
        if loan in loan.member.loans:
            loan.member.loans.remove(loan)
        if loan in self.active_loans:
            self.active_loans.remove(loan)
        Transaction("return", loan.member.get_member_id(), f"Book {loan.copy_obj.parent_book.title}", fine_amount)
        return days

# ------------------------------
# Auth Manager
# ------------------------------
class AuthManager:
    def __init__(self):
        self.logged_in = False
        self.current_user = None

    def login(self, member, pin):
        if member.is_locked():
            print("Account locked. Cannot login.")
            return False
        if member.verify_pin(pin):
            self.logged_in = True
            self.current_user = member
            print(f"Login successful. Welcome {member.get_name()}")
            Transaction("login", member.get_member_id())
            return True
        return False

    def logout(self):
        if self.logged_in:
            Transaction("logout", self.current_user.get_member_id())
            print(f"{self.current_user.get_name()} logged out.")
        self.logged_in = False
        self.current_user = None

# ------------------------------
# Report Generator
# ------------------------------
class ReportGenerator:
    def show_most_borrowed(self, books):
        print("\n--- Most Borrowed Books ---")
        if not books:
            print("No books in library.")
            return
        sorted_books = sorted(books, key=lambda b: b.borrow_count, reverse=True)
        for book in sorted_books[:3]:
            print(f"{book.title} - Borrowed: {book.borrow_count} times")

    def show_overdue(self, members):
        print("\n--- Overdue Books ---")
        any_overdue = False
        for member in members:
            for loan in member.loans:
                if loan.days_overdue() > 0:
                    any_overdue = True
                    print(f"{member.get_name()} has '{loan.copy_obj.parent_book.title}' overdue by {loan.days_overdue()} days")
        if not any_overdue:
            print("No overdue books right now.")

# ------------------------------
# Library System with CLI
# ------------------------------
class LibrarySystem:
    def __init__(self):
        self.admin = Admin("Admin")
        self.librarian = Librarian("Head Librarian")
        self.members = []
        self.books = []
        self.transactions = []
        self.auth = AuthManager()
        self.res_manager = ReservationManager()
        self.loan_manager = LoanManager(self.admin)
        self.reporter = ReportGenerator()
        self.load_data()

    # ------------------------------
    # File Handling
    # ------------------------------
    def save_data(self):
        try:
            members_data = []
            for m in self.members:
                members_data.append({
                    "id": m.get_member_id(),
                    "name": m.get_name(),
                    "contact": m.get_contact(),
                    "fines": m.fines,
                    "approved": m.approved,
                    "type": type(m).__name__
                })
            with open("members.json", "w") as f:
                json.dump(members_data, f)
            books_data = []
            for b in self.books:
                books_data.append({
                    "title": b.title,
                    "author": b.author,
                    "isbn": b.isbn,
                    "borrow_count": b.borrow_count,
                    "copies": [{"id": c.copy_id, "shelf": c.shelf, "status": "available" if c.available else "borrowed"} for c in b.copies],
                    "subject": b.subject,
                    "publisher": b.publisher,
                    "year": b.year
                })
            with open("books.json", "w") as f:
                json.dump(books_data, f)
        except Exception as e:
            print("Error saving data:", e)

    def load_data(self):
        if os.path.exists("members.json"):
            try:
                with open("members.json", "r") as f:
                    members_data = json.load(f)
                    for m in members_data:
                        member_type = m.get("type", "Member")
                        if member_type=="StudentMember":
                            member = StudentMember(m["name"], m.get("contact",""))
                        elif member_type=="FacultyMember":
                            member = FacultyMember(m["name"], m.get("contact",""))
                        elif member_type=="GuestMember":
                            member = GuestMember(m["name"], m.get("contact",""))
                        else:
                            member = Member(m["name"], m.get("contact",""))
                        member.fines = m.get("fines",0)
                        member.approved = m.get("approved", False)
                        self.members.append(member)
            except Exception as e:
                print("Error loading members:", e)
        if os.path.exists("books.json"):
            try:
                with open("books.json", "r") as f:
                    books_data = json.load(f)
                    for b in books_data:
                        book = Book(b["title"], b["author"], b["isbn"], b.get("subject",""), b.get("publisher",""), b.get("year",""))
                        book.borrow_count = b.get("borrow_count",0)
                        for c in b["copies"]:
                            copy_obj = BookCopy(c["id"], book, c.get("shelf","A1"))
                            copy_obj.available = True if c.get("status","available")=="available" else False
                            book.copies.append(copy_obj)
                        self.books.append(book)
            except Exception as e:
                print("Error loading books:", e)
    # ------------------------------
    # Display / Getter Helpers
    # ------------------------------
    def display_all_members(self):
        print("\n--- All Registered Members ---")
        print(f"{'ID':<38} {'Name':<20} {'Status':<10} {'Fines':<10}")
        print("-" * 85)
        
        if not self.members:
            print("No members found.")
            return
        status = ""
        for m in self.members:
            # We use ternary operators here to make status readable
            if m.is_locked():
              status = "Locked"
            elif m.is_approved():
              status = "Approved"
            else:
              status = "Pending"
            
            print(f"{m.get_member_id():<10} {m.get_name():<20} {status:<10} Rs.{m.fines:<10}")

    def display_all_books(self):
        print("\n--- Catalog ---")
        print(f"{'ISBN':<15} {'Title':<30} {'Author':<20} {'Copies (Avail/Total)'}")
        print("-" * 80)

        if not self.books:
            print("Library is empty.")
            return

        for book in self.books:
            total_copies = len(book.copies)
            avail_copies=0
            for copy in book.copies:
                if copy.available:
                    avail_copies+=1
            print(f"{book.isbn:<15} {book.title[:28]:<30} {book.author[:18]:<20} {avail_copies}/{total_copies}")

    def display_active_loans(self):
        print("\n--- Active Loans ---")
        loans = self.loan_manager.active_loans
        if not loans:
            print("No active loans.")
            return
            
        print(f"{'Loan ID':<38} {'Member':<15} {'Book Title':<30} {'Due Date'}")
        print("-" * 100)
        for loan in loans:
            print(f"{loan.loan_id:<10} {loan.member.get_name():<15} {loan.copy_obj.parent_book.title[:28]:<30} {loan.due_date.strftime('%Y-%m-%d')}")

    # ------------------------------
    # CLI Menu
    # ------------------------------
    def menu(self):
        while True:
            print("\n--- Welcome to The Library System ---")
            print("1. Register Member")
            print("2. Approve Member (Librarian)")
            print("3. Add Book")
            print("4. Borrow Book")
            print("5. Return Book")
            print("6. Renew Loan")
            print("7. Reserve Book")
            print("8. Pay Fine")
            print("9. Reports")
            print("10. View All Members") 
            print("11. View Catalog")
            print("12. View All Active Loans")
            print("0. Exit")
            
            choice = input("Enter choice: ")

            if choice == "1":
                # --- Validating Name (No Empty Strings, No Digits) ---
                while True:
                    name = input("Member Name: ").strip()
                    
                    if not name:
                        print("Error: Name cannot be empty.")
                    # Check if ANY character in the string is a number
                    
                    elif any(char.isdigit() for char in name):
                        print("Error: Name cannot contain numbers. Please enter a valid name.")
                    else:
                        break # Input is valid, exit the loop
                
            
                    
                # --- Validating Contact (Must be Integer as per your original code) ---
                while True:
                    try:
                        contact = int(input("Contact (Numbers only): "))
                        break
                    except ValueError:
                        print("Error: Contact must be a numeric integer.")

                # --- Validating Type (Must match options) ---
                while True:
                    mtype = input("Member Type (Student/Faculty/Guest): ").lower()
                    if mtype in ["student", "faculty", "guest"]:
                        break
                    print("Error: Invalid type. Please type Student, Faculty, or Guest.")

                # --- Validating PIN (Must be Integer) ---
                while True:
                    try:
                        pin = int(input("PIN (Numbers only): "))
                        break
                    except ValueError:
                        print("Error: PIN must be a number.")

                if mtype == "student":
                    m = StudentMember(name, contact, pin)
                elif mtype == "faculty":
                    m = FacultyMember(name, contact, pin)
                elif mtype == "guest":
                    m = GuestMember(name, contact, pin)
                else:
                    m = Member(name, contact, pin)
                
                self.members.append(m)
                print(f"Member registered: {m.get_name()} (ID: {m.get_member_id()})")
                self.save_data()

            elif choice == "2":
                mid = input("Member ID to approve: ")
                member = next((m for m in self.members if m.get_member_id() == mid), None)
                if member:
                    self.librarian.approve_member(member)
                    self.save_data()
                else:
                    print("Member not found.")

            elif choice == "3":
                # --- Validating Title ---
                while True:
                    title = input("Book Title: ").strip()
                    if title: break
                    print("Error: Title cannot be empty.")

                # --- Validating Author ---
                while True:
                    author = input("Author: ").strip()
                    if author and not author.isdigit(): 
                        break
                    print("Error: Author cannot be empty or just numbers.")

                # --- Validating ISBN (Must be Integer as per your original code) ---
                while True:
                    try:
                        isbn = int(input("ISBN (Numbers only): "))
                        break
                    except ValueError:
                        print("Error: ISBN must be a number.")

                # --- Validating Copies (Must be Integer) ---
                while True:
                    try:
                        copies = int(input("Number of copies: "))
                        if copies > 0: 
                            break
                        print("Error: Must add at least 1 copy.")
                    except ValueError:
                        print("Error: Number of copies must be an integer.")

                book = Book(title, author, isbn)
                for i in range(copies):
                    book.add_copy(f"{isbn}-C{i+1}")
                self.books.append(book)
                self.save_data()

            elif choice == "4":
                mid = input("Member ID: ")
                member = next((m for m in self.members if m.get_member_id() == mid), None)
                if not member:
                    print("Member not found.")
                    continue
                isbn = input("Book ISBN: ")

                book = next((b for b in self.books if str(b.isbn) == isbn), None) 
                
                if not book:
                    print("Book not found.")
                    continue
                copy_obj = book.available_copy()
                if not copy_obj:
                    print("No copies available.")
                    continue
                self.loan_manager.create_loan(member, copy_obj)
                self.save_data()

            elif choice == "5":
                mid = input("Member ID: ")
                member = next((m for m in self.members if m.get_member_id() == mid), None)
                if not member or not member.loans:
                    print("No loans found.")
                    continue
                print("Active loans:")
                for i, loan in enumerate(member.loans):
                    print(f"{i+1}. {loan.copy_obj.parent_book.title} - LoanID: {loan.loan_id}")
                
                # --- Validating Selection Index ---
                while True:
                    try:
                        idx = int(input("Select loan number to return: "))
                        if 1 <= idx <= len(member.loans):
                            break
                        print("Error: Invalid selection number.")
                    except ValueError:
                        print("Error: Please enter a number.")

                self.loan_manager.return_loan(member.loans[idx-1])
                self.save_data()

            elif choice == "6":
                mid = input("Member ID: ")
                member = next((m for m in self.members if m.get_member_id() == mid), None)
                if not member or not member.loans:
                    print("No loans found.")
                    continue
                print("Active loans:")
                for i, loan in enumerate(member.loans):
                    print(f"{i+1}. {loan.copy_obj.parent_book.title} - LoanID: {loan.loan_id}")
                
                # --- Validating Selection Index ---
                while True:
                    try:
                        idx = int(input("Select loan number to renew: "))
                        if 1 <= idx <= len(member.loans):
                            break
                        print("Error: Invalid selection number.")
                    except ValueError:
                        print("Error: Please enter a number.")

                member.loans[idx-1].renew()
                self.save_data()

            elif choice == "7":
                mid = input("Member ID: ")
                member = next((m for m in self.members if m.get_member_id() == mid), None)
                isbn = input("Book ISBN to reserve: ")
                # Comparing string input to stored isbn (likely int based on choice 3)
                book = next((b for b in self.books if str(b.isbn) == isbn), None)
                if member and book:
                    self.res_manager.create_reservation(book, member)
                    self.save_data()
                else:
                    print("Member or Book not found.")

            elif choice == "8":
                mid = input("Member ID: ")
                member = next((m for m in self.members if m.get_member_id() == mid), None)
                if member:
                    print(f"Current fines: Rs.{member.fines}")
                    
                    # --- Validating Payment Amount ---
                    while True:
                        try:
                            pay = float(input("Amount to pay: "))
                            if pay > 0:
                                break
                            print("Error: Amount must be positive.")
                        except ValueError:
                            print("Error: Please enter a valid decimal number.")

                    member.fines -= pay
                    if member.fines < 0: member.fines = 0
                    Transaction("pay_fine", member.get_member_id(), amount=pay)
                    print(f"Paid Rs.{pay}. Remaining fines: Rs.{member.fines}")
                    self.save_data()
                else:
                    print("Member not found.")

            elif choice == "9":
                self.reporter.show_most_borrowed(self.books)
                self.reporter.show_overdue(self.members)
            elif choice == "10":
                self.display_all_members()
            elif choice == "11":
                self.display_all_books()
            elif choice == "12":
                self.display_active_loans()
            elif choice == "0":
                print("Exiting system.")
                self.save_data()
                break
            else:
                print("Invalid choice.")
if __name__ == "__main__":
    lib_system = LibrarySystem()
    lib_system.menu()
    lib_system.menu()



--- Welcome to The Library System ---
1. Register Member
2. Approve Member (Librarian)
3. Add Book
4. Borrow Book
5. Return Book
6. Renew Loan
7. Reserve Book
8. Pay Fine
9. Reports
10. View All Members
11. View Catalog
12. View All Active Loans
0. Exit


Enter choice:  10



--- All Registered Members ---
ID                                     Name                 Status     Fines     
-------------------------------------------------------------------------------------
0bdc0401-427c-4c6f-ac28-64f8c42d0355 Moiz                 Pending    Rs.0.0       

--- Welcome to The Library System ---
1. Register Member
2. Approve Member (Librarian)
3. Add Book
4. Borrow Book
5. Return Book
6. Renew Loan
7. Reserve Book
8. Pay Fine
9. Reports
10. View All Members
11. View Catalog
12. View All Active Loans
0. Exit


Enter choice:  0


Exiting system.

--- Welcome to The Library System ---
1. Register Member
2. Approve Member (Librarian)
3. Add Book
4. Borrow Book
5. Return Book
6. Renew Loan
7. Reserve Book
8. Pay Fine
9. Reports
10. View All Members
11. View Catalog
12. View All Active Loans
0. Exit


Enter choice:  0


Exiting system.
